In [3]:
#Import packages for calculation
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import pynucastro as pyna

In [235]:

class MainphaseStar:
    #Stellar evolution simulator that opperates on the core conceit that the star remains in the Main Phase (where the primary energy in the star comes from hydrogen fusion) and remain in thermal and hydrostatic equilibrium


    def __init__(self, temp, dens, pres, lumin, pcenthydrogen, radius):
        #Takes 4 1d-arrays of equal size and 1 float.64
        self.t = temp #in kelvin
        self.d = dens #in gram/meters**3
        self.p = pres 
        self.L = lumin
        self.pcenthydrogen = pcenthydrogen #percent by mass of hydrogen for each layer
        self.r = radius #in meters
        #obtains the number of cells as well as the size of each cell 
        self.n = len(temp)
        self.rinc = self.r / self.n
        
        #use pynucastro to load reaction rates for p-p chain and c-n-o chain reactions, assuming the rates are bounded by the slowest reaction rate 
        rl = pyna.ReacLibLibrary()
        self.ppchainrate = rl.get_rate_by_name("p(p,)d")
        self.cnochainrate = rl.get_rate_by_name("n14(p,g)o15")

        #radiation constant as defined by Stefan Bolzman Law
        self.a = 7.5657 * 10 **(-16) * 6.241509*10**12
        #units MeV (m**-3) (K**-4)

        #gravitational constant
        self.G = 6.6743*10**-13
        #units m**3 /(g*s**2)

    def rincrinitialize(self):
        #reinitializes the calculation for the radius increment of each cell
        self.rinc = self.r / self.n
        self.rbycell = (self.n - np.asarray(range(0, self.n)))*self.rinc

    def massbyrad(self):
        #initializes mass array for each cell, where mass is the volume * the density
        self.mass = np.zeros_like(self.t) #Mass of interior of star from each cell
        self.layermass = np.zeros_like(self.t) #mass of material in the layer composed by each cell
        for i in range(0, self.n):
            self.layermass[i] = self.d[i]*( 4/3*np.pi*((self.n-i)*self.rinc)**3 - 4/3*((self.n - i - 1)*self.rinc)**3) 
        for i in range(0, self.n):
            self.mass[i] = np.sum(self.layermass[i:])

    def fusionenergy(self, density, perhydro, temp):
        #calculate fusion energy per unit mass based on temp and density of hydrogen- using p-p and c-n-o fusion paths 
        #Note: pynucastro assumes rho density is in grams/cm**3 thus we have to convert   
        
        if perhydro == 0 or temp == 0 or density == 0:
           return 0
        #evaluates reaction rates per cm^3 
        ppeval = self.ppchainrate[0].eval(temp, rho = (density*perhydro/(10**6)))
        cnoeval = self.cnochainrate.eval(temp, rho = (density*perhydro/(10**6)))

        #convert to reaction rates per unit mass
        ppeval = ppeval*10**6/density
        cnoeval = cnoeval*10**6/density

        #evaluate energy result per unit mass then converts mega electron volt to electron volt
        energyperunitmass = (ppeval*26.72 + cnoeval*25)*1000000 
        return energyperunitmass
    
    def drdm(self, m, r, rho):   
        #mass conservation equation
        if rho == 0:
            return 0
        
        return 1/(4*np.pi*r**2 *rho)

    def dpdm(self, m, p, r):
        #hydrostatic equilibrium
        if r == 0:
            return 0
        return -self.G*m/(4*np.pi**4*r)
    
    def dLdm(self, m, L, nuc):
        #energy conservation
        return -nuc
    
    def dTdm(self, m, T, r, P, L, dens):
        #energy transport
        if P == 0 or r == 0 or P == 0 or dens == 0 or L == 0 or m == 0:
            return 0
         
        k = 10**17*dens* np.sign(T) * (np.abs(T)) **(7/2)

        grad = ((3*k*L*P)/(16*np.pi*self.a*self.G*m*T**4))
        return -(self.G*m*T)/(4*np.pi*r**4*P)*grad 

        
    def massolve(self, masses):
        #Takes list of t masses to solve system at. First mass is assumed to be mass of original system 
        #returns a t by n array for each system quantity of values at each radius interval n for that mass
        #evaluates from center out for each time
        
        #initialize arrays
        self.massbyrad()
        self.rincrinitialize()
        temp = np.zeros((len(masses), self.n)) 
        dens = np.zeros((len(masses), self.n)) 
        pres = np.zeros((len(masses), self.n)) 
        radis = np.zeros((len(masses), self.n)) 
        pcent = np.zeros((len(masses), self.n)) 
        luminos = np.zeros((len(masses), self.n))
        efromrad = np.zeros((len(masses), self.n)) 
        intmass = np.zeros((len(masses), self.n))
        laymass = np.zeros((len(masses), self.n))
        temp[0, :] = self.t
        dens[0, :] = self.d
        pres[0, :] = self.p
        luminos[0, :] = self.L
        radis[0, :] = self.rbycell
        pcent[0, :] = self.pcenthydrogen
        intmass[0, :] = self.mass
        laymass[0, :] = self.layermass

        for q in range(1, len(masses)):

            #calculate the mass difference between this stage and previous mass
            massdif = masses[q] - masses[q-1]

            #evaluate per unit fusion energy at each cell and convert that into difference in mass across layers
            for i in range(0, self.n):
                efromrad[q-1, -i] = self.fusionenergy( float(dens[q-1, -i]), float(pcent[q-1, -i]), float(temp[q-1, -i]))
                if efromrad[q-1, -i] > 10**10 :
                    efromrad[q-1, -i] = 0
            pcentfusion = efromrad[q-1, :] / np.sum(efromrad[q-1, :])
            massdifbycell = massdif * pcentfusion
            laymass[q, :] = laymass[q-1, :] + massdifbycell

            for i in range(0, self.n):
                intmass[q, i] = sum(laymass[q, i:])
            #update hydrogen percentages
            pcent[q, :] = (pcent[q-1, :] * laymass[q-1, :] + massdifbycell) / laymass[q-1, :]

            #solve for radiuses
            for i in range(0, self.n):
                rho = sum(dens[q-1, i:])/(self.n - i)
                output = sp.integrate.solve_ivp(self.drdm, (laymass[q-1, i], laymass[q, i]), (radis[q-1, i],), args = (rho,))
                if i > 0:    
                    if output["y"][-1][-1] < radis[q, i -1]:
                        radis[q, i] = output["y"][-1][-1]
                else:
                    radis[q, i] = output["y"][-1][-1]

            #solve for preassures 
            for i in range(0, self.n):
                output = sp.integrate.solve_ivp(self.dpdm, (laymass[q-1, i], laymass[q, i]), (pres[q-1, i],), args = (radis[q, -i],))
                pres[q, i] = output["y"][-1][-1]
            
            #solve for luminosities
            for i in range(0, self.n):
                output = sp.integrate.solve_ivp(self.dLdm, (laymass[q-1, i], laymass[q, i]), (luminos[q-1, i],), args = (efromrad[q-1, i],))
                luminos[q, i] = output["y"][-1][-1]
            
            #solve for temperatures
            for i in range(0, self.n):
                output = sp.integrate.solve_ivp(self.dTdm, (intmass[q-1, i], intmass[q, i]), (temp[q-1, i],), args = (radis[q, i], pres[q, i], luminos[q, i], dens[q-1, i]))
                temp[q, i] = output["y"][-1][-1]
                

            #density wrong!   
            for i in range(0, self.n-1):
                dens[q, i] = laymass[q, i]/(4/3*np.pi*radis[q, i]**3- 4/3*np.pi*radis[q, i+1]**3)
            

        return (temp, dens, pres, luminos, pcent, radis, intmass, laymass, efromrad)

In [236]:
#1 solar mass run from MESA simulation start point
temp = np.asarray(range(0, 150))/150*(10**7.1346983)
dens = np.append(np.logspace(-6.7, 1.8, 10), np.logspace(1.8, 1.891, 140))*1000
pres =   np.asarray(range(0, 150)) * (3.4*10**11)/150*101.325*1000
radis = 700000*1000
lumin = np.logspace(-99, -0.55634, 150)
pcent = np.ones(150)*0.698965
star = MainphaseStar(temp, dens, pres, lumin, pcent, radis)

In [273]:
star.rincrinitialize()
star.massbyrad()
star.mass
masspoints = np.linspace(1.994401955*10**33, 1.994401955*10**33*0.9999991, 80)
output = star.massolve(masspoints)

array([1.97233183e+23, 1.70112436e+24, 1.46701250e+25, 1.26494734e+26,
       1.09056398e+27, 9.40089300e+27, 8.10262024e+28, 6.98263625e+29,
       6.01658477e+30, 5.18342065e+31, 5.07426691e+31, 4.97414899e+31,
       4.87525429e+31, 4.77757800e+31, 4.68111529e+31, 4.58586125e+31,
       4.49181095e+31, 4.39895940e+31, 4.30730156e+31, 4.21683234e+31,
       4.12754662e+31, 4.03943922e+31, 3.95250490e+31, 3.86673838e+31,
       3.78213435e+31, 3.69868741e+31, 3.61639214e+31, 3.53524307e+31,
       3.45523467e+31, 3.37636137e+31, 3.29861753e+31, 3.22199748e+31,
       3.14649549e+31, 3.07210579e+31, 2.99882253e+31, 2.92663984e+31,
       2.85555179e+31, 2.78555239e+31, 2.71663561e+31, 2.64879534e+31,
       2.58202546e+31, 2.51631976e+31, 2.45167201e+31, 2.38807588e+31,
       2.32552505e+31, 2.26401308e+31, 2.20353353e+31, 2.14407987e+31,
       2.08564554e+31, 2.02822391e+31, 1.97180831e+31, 1.91639199e+31,
       1.86196817e+31, 1.80853000e+31, 1.75607059e+31, 1.70458297e+31,
      